![Logo SENATI](logo_senati.png)

**SENATI | Escuela de Tecnologías de la Información**  
**Taller: Machine Learning y Deep Learning**  
*Instructor: Ing. Gunar Castro Iñigo*

---

**Trabajo desarrollado por:** COCA CCORPA LORENA ABIGAIL  
**Institución:** SENATI · 2026  
**Actividad:** Sistema de estimación de costo de transporte agrícola  
**Nota:** Los datos suministrados por el taller son simulados; los resultados y pruebas son cálculos realizados sobre esos CSV.

# Sistema de estimación de costo de transporte agrícola — caso práctico intermedio

**Contexto:** Un transportista quiere estimar el costo de un viaje a partir de distancia, peso y tipo de vía. Datos simulados inspirados en el transporte agrícola regional.

**Objetivo:** aplicar regresión múltiple desde lectura de datos hasta una web pequeña. Aquí no se entrega el resultado completo: debes demostrar lo que aprendiste durante los cuatro proyectos.

## Paso 1. Explorar el CSV
Busca las columnas de entrada `X`: distancia_km, carga_kg, via_no_asfaltada, y la respuesta `y`: `costo_transporte_soles`.

**¿Qué podemos preguntar al grupo?** ¿Qué significa cada variable en la vida real? ¿Cuál sería el peligro de usar esta predicción sin verificarla?

### Respuestas cortas

- `distancia_km`: kilómetros del viaje; `carga_kg`: peso transportado; `via_no_asfaltada`: 1 si la ruta no está asfaltada. La respuesta es el costo aproximado en soles.
- **¿Cuál es el peligro de no verificarla?** Tomar decisiones basándose en un cálculo equivocado, especialmente porque los datos del curso son simulados.

<div style="background-color:#FFF0A8;color:#463600;border-left:7px solid #D6A400;padding:14px 18px;border-radius:9px;margin:10px 0 16px"><strong>🟨 INICIO DEL DESARROLLO DE LA ACTIVIDAD</strong><br>Desde aquí se realiza la lectura del CSV y el entrenamiento del modelo. Se conserva el enunciado original y el código se ejecuta en orden.</div>

In [1]:
import pandas as pd

datos = pd.read_csv('datos.csv')
print(datos.head())
print(datos.isna().sum())

   viaje  distancia_km  carga_kg  via_no_asfaltada  costo_transporte_soles
0  R-001         140.1      1775                 0                  592.32
1  R-002         131.9       425                 1                  442.18
2  R-003         102.8      1525                 0                  478.20
3  R-004          50.5       325                 1                  221.73
4  R-005          67.1       975                 0                  314.59
viaje                     0
distancia_km              0
carga_kg                  0
via_no_asfaltada          0
costo_transporte_soles    0
dtype: int64


## Paso 2. Entrenar (completar)
Adapta el procedimiento de las prácticas 2 y 4. No copies resultados de otros compañeros: explica qué variables usaste, cuántas filas reservaste y cómo interpretas el MAE.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

columnas = ['distancia_km', 'carga_kg', 'via_no_asfaltada']
objetivo = 'costo_transporte_soles'
X = datos[columnas]
y = datos[objetivo]

# Entrenamiento y evaluación (80/20, random_state=42, sin fuga de datos)
X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)
modelo = LinearRegression()
modelo.fit(X_ent, y_ent)
estimaciones = modelo.predict(X_prueba)
mae = mean_absolute_error(y_prueba, estimaciones)
r2 = r2_score(y_prueba, estimaciones)
print('Registros:', len(datos),'| Entrenamiento:', len(X_ent),'| Prueba:',len(X_prueba))
print('MAE:', round(mae,3),'| R²:',round(r2,4))
print('Intercepto:',round(modelo.intercept_,4))
for nombre, peso in zip(columnas,modelo.coef_):
    print(f'{nombre}: {peso:.5f}')
print('MAE de referencia (media de entrenamiento):',round(mean_absolute_error(y_prueba, [y_ent.mean()]*len(y_prueba)),3))

Registros: 130 | Entrenamiento: 104 | Prueba: 26
MAE: 17.794 | R²: 0.9772
Intercepto: 21.5196
distancia_km: 2.77424
carga_kg: 0.10729
via_no_asfaltada: 26.44186
MAE de referencia (media de entrenamiento): 123.245


## Paso 3. Aplicación web
Tu página necesita título realista, 3 campos con unidades, botón “Calcular”, resultado y una advertencia de estimación. Usa Flask, HTML y CSS (sin JavaScript). Separa los archivos en las carpetas habituales.

**Aplicación preparada:** `app.py`, `templates/index.html` y `static/estilos.css`. El formulario usa los tres datos de entrada, muestra una estimación y aclara que el cálculo es educativo. La comprobación del sitio en navegador queda por registrar como evidencia.

## Paso 4. Pruebas obligatorias
- Viaje 25 km, 400 kg, ruta asfaltada
- Viaje 100 km, 1200 kg, ruta no asfaltada

¿Qué valores devolvió tu sistema? ¿Tienen sentido? ¿En qué circunstancias la predicción podría fallar?

<div style="background-color:#FFF0A8;color:#463600;border-left:7px solid #D6A400;padding:14px 18px;border-radius:9px;margin:10px 0 16px"><strong>🟨 INICIO DEL DESARROLLO DE LA ACTIVIDAD</strong><br>Desde aquí se realiza las dos pruebas solicitadas. Se conserva el enunciado original y el código se ejecuta en orden.</div>

## Resultados, pruebas obligatorias y conclusiones

In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
pruebas = pd.DataFrame([(25, 400, 0), (100, 1200, 1)],columns=columnas)
pruebas['prediccion'] = modelo.predict(pruebas[columnas])
print(pruebas.to_string(index=False))
resultado_prueba = X_prueba.copy()
resultado_prueba['real'] = y_prueba.values
resultado_prueba['estimado'] = estimaciones
resultado_prueba.to_csv('predicciones_prueba.csv',index=False)
with open('modelo.json','w',encoding='utf-8') as archivo:
    json.dump({'intercepto':float(modelo.intercept_),'coeficientes':[float(v) for v in modelo.coef_], 'columnas':columnas,'objetivo':objetivo},archivo,indent=2,ensure_ascii=False)
print('Guardados modelo.json y predicciones_prueba.csv')

 distancia_km  carga_kg  via_no_asfaltada  prediccion
           25       400                 0  133.790051
          100      1200                 1  454.128606
Guardados modelo.json y predicciones_prueba.csv


### Respuestas cortas

**Prueba 1:** 25 km, 400 kg, ruta asfaltada → **S/ 133.79**.  
**Prueba 2:** 100 km, 1200 kg, ruta no asfaltada → **S/ 454.13**.  
**¿Tienen sentido?** Sí, el viaje largo y con más carga cuesta más.  
**¿Cuándo puede fallar?** Si cambia el precio del combustible o aparecen peajes, clima y costos no registrados.

## Evidencias y evaluación
Entregar notebook ejecutado, CSV, script de entrenamiento, `modelo.json`, aplicación Flask, dos capturas de prueba y conclusiones propias. Criterios: datos 3 pts; modelo 5 pts; métricas 4 pts; aplicación 5 pts; documentación 3 pts.

**Interpretación:** El MAE es el error absoluto medio en las unidades de la variable objetivo; R² mide el ajuste frente a la referencia de la media. Los coeficientes describen asociaciones dentro de los datos simulados, no causalidad. **Limitaciones:** datos sintéticos, posibles cambios en costos y demanda, variables omitidas; la validez requiere datos reales y evaluación externa. Las dos pruebas pertenecen al enunciado original.